In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
import os

#########################################################################
if torch.cuda.is_available():
    #### Run on Gogle Colab
    # !pip install torch-ort
    # !python -m torch_ort.configure
    !python -m pip install lightning
    # !python -m pip install imbalanced-learn
    # pin_memory = False
    # num_workers = os.cpu_count() - 2 if os.cpu_count() -2 > 3 else 3

    device = torch.device("cuda")
    accelerator = "gpu"


    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = "/content/drive/MyDrive/Colab Data/RetrievalModel"
    # root_dir = "./drive/MyDrive/Colab Data/ResidualNeuralNetwork"
    torch.cuda.memory.empty_cache()
else:
    ### Run on my personal tower
    device = torch.device("cpu")
    accelerator = "cpu"
    data_dir = "EmbeddedData"
    # root_dir = "."

torch.set_float32_matmul_precision("medium")




#########################################################################
class Hyperparameters:
    ######
    lr = 1e-3
    l2 = 1e-5
    lr_decay_gamma = 0.1
    lr_decay_stepsize = 2
    lr_decay_threshold = 1e-4
    lr_decay_parameter = "train_loss"
    similarity_eps = 1e-10
    gradient_clip_val = 1.0
    Epochs = 30
    earlyStopping_min_delta = 1e-8
    earlyStopping_patience = 7
    
    Project_Name = "BERT_Retrieval"
    modelname = "BERT_WordEmbedding"
    checkPoint_BasePath = f"Training_Progress/Checkpoints/{Project_Name}"
    logging_BasePath = f"Training_Progress/Logs/{Project_Name}"
    bestmodel_BasePath = f"Training_Progress/BestModel/{Project_Name}"
    tensor_size = 32

    bert_batch_size = 256 if torch.cuda.is_available() else 32
    # bert_embedding_size = 100
    # bert_nheads = 4
    # bert_vocab_size = 30522 # comes from the pre-trained BERT word-tokenizer

hp = Hyperparameters()
for i in [hp.modelname]:
    os.makedirs(f"{hp.checkPoint_BasePath}/{i}", exist_ok = True)
    os.makedirs(f"{hp.logging_BasePath}/{i}", exist_ok = True)
    os.makedirs(f"{hp.bestmodel_BasePath}/{i}", exist_ok = True)


##########################################################################################


import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

formatter = logging.Formatter('%(name)s : %(levelname)s:%(levelno)s  %(asctime)s    %(message)s ')

info_handler = logging.FileHandler(f"{hp.logging_BasePath}/info.log") # default mode is already 'append'
info_handler.setLevel(logging.INFO)
info_handler.setFormatter(formatter)
logger.addHandler(info_handler)

In [ ]:
esci_encode_dict = {
    "E": 0,
    "S": 1,
    "C": 2,
    "I": 3
}

esci_decode_dict = {i:j for j, i in esci_encode_dict.items()}

In [ ]:
id_train = pd.read_parquet(f"{data_dir}/WordEmbedding_inputs_train.parquet").reset_index(drop = True)
y_train = pd.read_parquet(f"{data_dir}/WordEmbedding_esci_train.parquet").map(lambda x: esci_encode_dict[x]).reset_index(drop = True).squeeze()
mask_train = pd.read_parquet(f"{data_dir}/WordEmbedding_masks_train.parquet").reset_index(drop = True)

columns_with_info = ["product_id", "query_id", "example_id", "split", "esci_label"]
id_valid = pd.read_parquet(f"{data_dir}/WordEmbedding_inputs_valid.parquet").reset_index(drop = True)
y_valid = id_valid.loc[:, "esci_label"].map(lambda x: esci_encode_dict[x])
id_valid = id_valid.drop(columns = columns_with_info)
mask_valid = pd.read_parquet(f"{data_dir}/WordEmbedding_masks_valid.parquet").reset_index(drop = True).drop(columns = columns_with_info)

# Look at the data

In [ ]:
train_classes, train_counts = np.unique(y_train, return_counts = True)
valid_classes, valid_counts = np.unique(y_valid, return_counts = True)

train_classes_esci = [esci_decode_dict[x] for x in train_classes]
valid_classes_esci = [esci_decode_dict[x] for x in valid_classes]

fig = plt.figure(figsize = (10, 5))
ax = fig.add_subplot(121)
ax.bar(train_classes_esci, train_counts)
ax.set_title("Training Classes\n(Undersampled)")
ax = fig.add_subplot(122)
ax.bar(valid_classes_esci, valid_counts)
ax.set_title("Validation Classes\n(true representaion of class distribution)")
plt.show()

## Class weights for training

In [ ]:
total_samples = y_train.shape[0]
num_classes = len(train_counts)
class_weights = total_samples / (num_classes * train_counts)
class_weights = torch.tensor(class_weights / np.sum(class_weights)).type(torch.float)
print(class_weights)

# Test the Dataset and the Model (how does CLS look in praktice)

import SimpleBERT#  import Dataset_WordEmbedding, SimpleBERTModel
from torch.utils.data import Dataset, DataLoader

import importlib
importlib.reload(SimpleBERT)




train_dataset = SimpleBERT.Dataset_WordEmbedding(hp, id_train, mask_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)

pii, qii, pam, qam, true_label = next(iter(train_dataloader))

embed_func = torch.nn.Embedding(num_embeddings = vocab_size, embedding_dim = embedding_size)

pii_embedded = embed_func(pii).permute(1, 0, 2)

testmodel = SimpleBERT.SimpleBERTModel(in_nodes = embedding_size, nhead = 4)
output = testmodel(pii_embedded, pam)

output.shape

# Implement the Model

## Dataset and Dataloader

In [ ]:
import PreTrainedBERT
from torch.utils.data import Dataset, DataLoader
import gc



train_dataset = PreTrainedBERT.Dataset_WordEmbedding(
    hp = hp,
    input_ids = id_train,
    attention_masks = mask_train,
    esci_labelencoded = y_train)

del id_train
del mask_train
del y_train

val_dataset = PreTrainedBERT.Dataset_WordEmbedding(
    hp = hp,
    input_ids = id_valid,
    attention_masks = mask_valid,
    esci_labelencoded = y_valid
)

del mask_valid
del id_valid
del y_valid

gc.collect()

train_dataloader = DataLoader(train_dataset, batch_size = hp.bert_batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = hp.bert_batch_size, shuffle = False)

## The Training

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning import Trainer, callbacks, loggers


# import importlib
# importlib.reload(PreTrainedBERT)


earlyStop = callbacks.EarlyStopping(monitor = 'val_loss',
                                    patience = hp.earlyStopping_patience,
                                    min_delta = hp.earlyStopping_min_delta)

checkPointTraining = callbacks.ModelCheckpoint(dirpath = f"{hp.checkPoint_BasePath}/{hp.modelname}",
                                                filename = "{epoch}",
                                                monitor='val_loss',
                                                verbose=0,
                                                save_top_k = -1,
                                                save_weights_only=False)

csv_logger = loggers.CSVLogger(save_dir = hp.logging_BasePath, name = hp.modelname)

# The Trainer
trainer = Trainer(callbacks=[earlyStop, checkPointTraining],
                  logger = csv_logger,
                  max_epochs = hp.Epochs,
                  accelerator = accelerator,
                  devices = 1,
                  gradient_clip_val = hp.gradient_clip_val
                 )


checkpoints = [i for i in os.listdir(f"{hp.checkPoint_BasePath}/{hp.modelname}")]
if ".training_done" in checkpoints:
    print("Model has already been fully trained.")
else:
    checkpoints = [i for i in checkpoints if i.endswith(".ckpt")]
    if checkpoints:
        starting_epoch = np.max([int(i.replace(".ckpt", "").replace("epoch=", "")) for i in checkpoints])
        last_checkpoint_filepath = f"{hp.checkPoint_BasePath}/{hp.modelname}/epoch={starting_epoch}.ckpt"
        #### Apparently Google Colab can cause some sort of weird config in the pytorch safefile as it is honed for tensorflow. This causes the weights_only ckeckpoint to not be accepted when loading it
        # the serialization options allow the pytorch weights_only checkpoint to be loaded anyway
        torch.serialization.add_safe_globals([np.dtypes.Int64DType])
        with torch.serialization.safe_globals([SimpleBERT.Retrieval_Lightning, np._core.multiarray.scalar, np.dtype]):
            Model = PreTrainedBERT.Retrieval_Lightning.load_from_checkpoint(last_checkpoint_filepath, class_weights = class_weights, device = device, hp = hp).to(device)
        trainer.fit(model = Model,
                    train_dataloaders = train_dataloader,
                    val_dataloaders = val_dataloader,
                    ckpt_path=last_checkpoint_filepath
                   )
    else:
        Model = PreTrainedBERT.Retrieval_Lightning(class_weights = class_weights, device = device, hp = hp).to(device)
        trainer.fit(model = Model,
                    train_dataloaders = train_dataloader,
                    val_dataloaders = val_dataloader
                   )

In [ ]:
from pathlib import Path
Path(f"{hp.checkPoint_BasePath}/{modelname}/.training_done").touch()

os.makedirs(f"{hp.bestmodel_BasePath}/{modelname}", exist_ok = True)
if "BestModel.save" not in os.listdir(f"{hp.bestmodel_BasePath}/{modelname}"):
    Model = SimpleBERT.Retrieval_Lightning.load_from_checkpoint(checkPointTraining.best_model_path, weights = class_weights, device = device, num_classes = num_classes).to(device)
    saving_Trainer = Trainer(max_epochs = 0)
    saving_Trainer.fit(Model, train_dataloaders = SentenceEmbedding_Train, val_dataloaders = SentenceEmbedding_Val)
    saving_Trainer.save_checkpoint(f"{hp.bestmodel_BasePath}/{modelname}/BestModel.save", weights_only = True)